# AI Recommendation Module — Full Pipeline

This notebook walks through the full pipeline:
1. Data loading & cleaning
2. Exploratory Data Analysis (EDA)
3. Market Basket Analysis (Apriori)
4. AutoEncoder training
5. NCF training
6. LSTM training
7. Evaluation
8. Sample predictions

In [ ]:
import os, sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config import *

np.random.seed(SEED)
%matplotlib inline

## 1. Data Loading & Cleaning

In [ ]:
from preprocessing import load_and_clean, build_customer_item_matrix, build_invoice_baskets, build_purchase_sequences, build_mappings, save_popular_items

df = load_and_clean()
print(f"Clean rows: {len(df)}")
print(f"Unique customers: {df['CustomerID'].nunique()}")
print(f"Unique products: {df['StockCode'].nunique()}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Purchase distribution per customer
purchases_per_customer = df.groupby('CustomerID')['StockCode'].nunique()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(purchases_per_customer, bins=50, edgecolor='black')
axes[0].set_title('Unique Products per Customer')
axes[0].set_xlabel('Number of unique products')
axes[0].set_ylabel('Number of customers')

# Top 20 products
top_products = df['Description'].value_counts().head(20)
axes[1].barh(range(len(top_products)), top_products.values)
axes[1].set_yticks(range(len(top_products)))
axes[1].set_yticklabels(top_products.index, fontsize=8)
axes[1].set_title('Top 20 Products by Frequency')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"Avg products per customer: {purchases_per_customer.mean():.1f}")
print(f"Median products per customer: {purchases_per_customer.median():.1f}")

In [ ]:
# Monthly sales trend
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
monthly = df.groupby('YearMonth').agg(orders=('InvoiceNo','nunique'), revenue=('Quantity','sum')).reset_index()
monthly['YearMonth'] = monthly['YearMonth'].astype(str)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(monthly['YearMonth'], monthly['orders'], alpha=0.7, label='Orders')
ax1.set_ylabel('Number of Orders')
ax1.set_xlabel('Month')
plt.xticks(rotation=45)
ax2 = ax1.twinx()
ax2.plot(monthly['YearMonth'], monthly['revenue'], color='red', marker='o', label='Units Sold')
ax2.set_ylabel('Units Sold')
plt.title('Monthly Sales Trend')
plt.tight_layout()
plt.show()

## 3. Build Matrices & Mappings

In [ ]:
customer_item_matrix = build_customer_item_matrix(df)
basket_matrix = build_invoice_baskets(df)
sequences = build_purchase_sequences(df)
mappings = build_mappings(df)
save_popular_items(df)

import pickle
with open(MAPPINGS_PATH, 'wb') as f:
    pickle.dump(mappings, f)

print(f"Customer-item matrix: {customer_item_matrix.shape}")
print(f"Basket matrix: {basket_matrix.shape}")
print(f"Sequences: {len(sequences)}")
print(f"Users: {mappings['n_users']}, Items: {mappings['n_items']}")

## 4. Market Basket Analysis (Apriori)

In [ ]:
from apriori_rules import run_apriori
rules = run_apriori(basket_matrix)
if len(rules) > 0:
    print(rules[['antecedents','consequents','support','confidence','lift']].head(10))

## 5. Data Split

In [ ]:
all_users = list(customer_item_matrix.index)
np.random.shuffle(all_users)
n = len(all_users)
train_users = all_users[:int(n*0.8)]
val_users = all_users[int(n*0.8):int(n*0.9)]
test_users = all_users[int(n*0.9):]
print(f"Train: {len(train_users)}, Val: {len(val_users)}, Test: {len(test_users)}")

matrix_values = customer_item_matrix.values.astype(np.float32)
user_idx_map = {u: i for i, u in enumerate(customer_item_matrix.index)}
train_idx = [user_idx_map[u] for u in train_users if u in user_idx_map]
val_idx = [user_idx_map[u] for u in val_users if u in user_idx_map]
train_matrix = matrix_values[train_idx]
val_matrix = matrix_values[val_idx]

## 6. Train AutoEncoder

In [ ]:
from model_autoencoder import train_autoencoder
ae_model, ae_history = train_autoencoder(train_matrix, val_matrix, mappings['n_items'])

## 7. Train NCF

In [ ]:
from model_ncf import train_ncf
ncf_model, ncf_history = train_ncf(customer_item_matrix, mappings, train_users)

## 8. Train LSTM

In [ ]:
from model_lstm import train_lstm
lstm_model, lstm_history = train_lstm(sequences, mappings, train_users)

## 9. Evaluation

In [ ]:
from evaluate import evaluate_model, build_comparison_table, plot_comparison
from inference import RecommendationEngine

engine = RecommendationEngine()
engine.ae_model = ae_model
engine.ncf_model = ncf_model
engine.lstm_model = lstm_model
engine.mappings = mappings
engine.customer_vectors = customer_item_matrix
engine.popular_items = pickle.load(open(POPULAR_PATH, 'rb'))

train_cim = customer_item_matrix.loc[customer_item_matrix.index.isin(train_users)]
eval_users = test_users[:100]
all_results = {}

def ae_score_fn(uid):
    vec = engine._get_user_vector(uid)
    scores = engine._ae_scores(vec)
    return {mappings['idx2item'][i]: float(scores[i]) for i in range(mappings['n_items'])}

all_results['AutoEncoder'] = evaluate_model('AutoEncoder', ae_score_fn, eval_users, customer_item_matrix, train_cim, mappings)

def ncf_score_fn(uid):
    scores = engine._ncf_scores(uid)
    return {mappings['idx2item'][i]: float(scores[i]) for i in range(mappings['n_items'])}

all_results['NCF'] = evaluate_model('NCF', ncf_score_fn, eval_users, customer_item_matrix, train_cim, mappings)

def lstm_score_fn(uid):
    items = df[df['CustomerID'] == uid].sort_values('InvoiceDate')['StockCode'].tolist()
    scores = engine._lstm_scores(items)
    return {mappings['idx2item'][i]: float(scores[i]) for i in range(mappings['n_items'])}

all_results['LSTM'] = evaluate_model('LSTM', lstm_score_fn, eval_users, customer_item_matrix, train_cim, mappings)

def ensemble_score_fn(uid):
    items = df[df['CustomerID'] == uid].sort_values('InvoiceDate')['StockCode'].tolist()
    vec = engine._get_user_vector(uid, items)
    combined = (engine._ae_scores(vec) + engine._ncf_scores(uid) + engine._lstm_scores(items)) / 3.0
    return {mappings['idx2item'][i]: float(combined[i]) for i in range(mappings['n_items'])}

all_results['Ensemble'] = evaluate_model('Ensemble', ensemble_score_fn, eval_users[:50], customer_item_matrix, train_cim, mappings)

table = build_comparison_table(all_results)
plot_comparison(all_results)

## 10. Sample Predictions

In [ ]:
for uid in test_users[:3]:
    items = df[df['CustomerID'] == uid].sort_values('InvoiceDate')['StockCode'].tolist()[:5]
    recs, strategy = engine.recommend(uid, items, top_n=5)
    print(f"\nUser: {uid}")
    print(f"  Input items: {items}")
    print(f"  Strategy: {strategy}")
    for r in recs:
        print(f"    -> {r['product_id']}: {r['description']} (score: {r['score']})")